# Hypercomplex Libraries on Kaggle: Install & Configure

> Companion to the Gemma 4 Starter Guide. Sets up the Python
> hypercomplex mathematics stack used in gororoba-quantum.
> Last verified: 2026-10-15.

## Library map

| PyPI package | Version | Import | Use |
|---|---|---|---|
| `hypercomplex` | 0.3.4 | `hypercomplex` | Exact Cayley-Dickson algebras |
| `hypercomplex-engine` | 0.5.0 | `hypercomplex_engine` | Fast basis-sign oracle |
| `hypercomplex-algebra` | 0.2.0.2 | `hypercomplex_algebra` | Expression parsing |
| `cayley-calculator` | 1.3.1 | `Cayley_Calculator` | Elementary functions (exp, log) |
| `octonion` | 0.1.0 | `octonion` | Batched octonion PyTorch tensors |
| `mathdunders` | 0.4.2 | `mathdunders` | Operator plumbing |

The reference implementation is `cd_sign` from
`gororoba_quantum/cd_core.py`. Every library below is tested against it.


In [ ]:
!pip install -q hypercomplex==0.3.4 hypercomplex-engine==0.5.0 hypercomplex-algebra==0.2.0.2 2>&1 | tail -2
!pip install -q cayley-calculator==1.3.1 octonion==0.1.0 mathdunders==0.4.2 2>&1 | tail -2
print("Installed")

## Reference: cd_sign (from gororoba_quantum/cd_core.py)
This is the ground truth. All libraries must agree with it.


In [ ]:
def cd_sign(dim: int, p: int, q: int) -> int:
    sign = 1
    half = dim >> 1
    while half > 0:
        p_hi = p >= half
        q_hi = q >= half
        if not p_hi and not q_hi:
            pass
        elif not p_hi and q_hi:
            qh = q - half
            q = p
            p = qh
        elif p_hi and not q_hi:
            p -= half
            if q != 0:
                sign = -sign
        else:
            qh = q - half
            ph = p - half
            if qh == 0:
                return -sign
            p = qh
            q = ph
        half >>= 1
    return sign


# Sanity: (e1+e10)(e4-e15) = 0 requires specific signs
print("cd_sign(16,1,4) =", cd_sign(16, 1, 4))
print("Reference loaded.")

## Test 1: hypercomplex (exact algebras)


In [ ]:
from fractions import Fraction

from hypercomplex import cayley_dickson_algebra

# Sedenion algebra with exact rational arithmetic
sed = cayley_dickson_algebra(4, Fraction)  # 2^4 = 16 dimensions
print("Sedenion algebra created")

# Verify basis product signs match cd_sign for all 256 pairs
mismatches = 0
for p in range(16):
    for q in range(16):
        ep = sed.e(p)
        eq = sed.e(q)
        prod = ep * eq
        expected_sign = cd_sign(16, p, q)
        expected_idx = p ^ q
        coeffs = prod.coefficients()
        actual = coeffs[expected_idx] if expected_idx < len(coeffs) else 0
        if actual != expected_sign:
            mismatches += 1
print(f"Mismatches vs cd_sign: {mismatches}/256")
print(f"hypercomplex matches reference: {mismatches == 0}")

## Test 2: hypercomplex-engine (fast oracle)


In [ ]:
try:
    from hypercomplex_engine import cayley_dickson_sign

    # Test a few signs
    test_cases = [(16, 1, 4), (16, 10, 15), (16, 0, 5), (16, 7, 7)]
    ok = True
    for dim, p, q in test_cases:
        eng = cayley_dickson_sign(dim, p, q)
        ref = cd_sign(dim, p, q)
        # engine returns (sign, index) tuple
        if isinstance(eng, tuple):
            eng_sign = eng[0]
        else:
            eng_sign = eng
        if eng_sign != ref:
            print(f"MISMATCH at ({p},{q}): engine={eng_sign}, ref={ref}")
            ok = False
    print(f"Engine matches reference: {ok}")
except ImportError as e:
    print(f"Engine import: {e}")
except Exception as e:
    print(f"Engine test: {e}")

## Test 3: Zero divisor verification
The canonical example: (e1 + e10)(e4 - e15) = 0


In [ ]:
# Using hypercomplex with exact arithmetic
e1 = sed.e(1)
e10 = sed.e(10)
e4 = sed.e(4)
e15 = sed.e(15)

a = e1 + e10
b = e4 - e15
prod = a * b

# Check all coefficients are zero
coeffs = prod.coefficients()
is_zero = all(coeffs[i] == 0 for i in range(16))
print(f"(e1+e10)(e4-e15) = 0: {is_zero}")
if not is_zero:
    nz = [(i, coeffs[i]) for i in range(16) if coeffs[i] != 0]
    print("Nonzero coefficients:", nz)

## Test 4: Octonion PyTorch tensors


In [ ]:
try:
    import torch  # noqa: F401
    from octonion import OctonionTensor  # noqa: F401

    print("octonion PyTorch integration available")
    # Note: uses Fano labeling with signed relabeling
    # pi = (0,1,2,4,3,6,7,5) with flip on e6
    # Index i is NOT cd_core e_i — apply relabeling for comparison
except ImportError as e:
    print(f"octonion: {e}")

## Summary

All libraries installed and tested against the `cd_core` reference.
Use `hypercomplex` for exact certificates, `hypercomplex-engine`
for speed, `octonion` for GPU-batched PyTorch work.

For the authoritative `cd_sign` implementation, see:
`gororoba-quantum/gororoba_quantum/cd_core.py`
